In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

import os
from neo4j import GraphDatabase
from src.rag_pipeline import build_vector_store, answer_question

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\DELL\DataScience\envs\ds_env\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
uri = os.getenv("NEO4J_URI", "bolt://127.0.0.1:7687")
user = os.getenv("NEO4J_USER", "neo4j")
password = os.getenv("NEO4J_PASSWORD", "enterprise-kg")

driver = GraphDatabase.driver(uri, auth=(user, password))
print("Connected to Neo4j")

Connected to Neo4j


In [3]:
index, companies = build_vector_store(driver)

Building vector store from 1375 company descriptions...


Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Vector store built. 1375 vectors indexed.


In [4]:
from src.rag_pipeline import retrieve_context

query = "Which companies work in social media and were acquired?"
context = retrieve_context(query, index, companies)
print(context)

Company: Socialthing
Description: Simplifying the management of sites and services that you already use like blogs and social networks, socialthing! synchronizes your personal information, content, and friendships so that you can post this information across the social web, as opposed to just having it contained in a single site.Socialthing is a 2007 TechStars company. Socialthing went into private alpha on January 15th, 2008 and launched their private beta at SXSW in March 2008. Socialthing was acquired by AOL.  While the price is unconfirmed, the acquisition value is rumored to be between $5M - $10M.

Company: Vitrue
Description: Vitrue offers the industry’s most scalable and secure cloud-based social marketing solution, enabling brands to harness the full marketing potential of social on Facebook, Twitter, Google+, YouTube and emerging platforms. The award-winning Vitrue SRM platform collectively manages more than one billion social relationships in over 100 countries across more th

In [5]:
query = "Tell me about companies acquired by Microsoft"
answer, context = answer_question(query, index, companies)

print("CONTEXT RETRIEVED:")
print(context)
print("\nANSWER:")
print(answer)

CONTEXT RETRIEVED:
Company: Microsoft
Description: Microsoft is an American multinational corporation that develops, manufactures, licenses, supports and sells computer software, consumer electronics and personal computers and services. Its best known software products are the Microsoft Windows line of operating systems, Microsoft Office office suite, and Internet Explorer web browser. Its flagship hardware products are Xbox game console and the Microsoft Surface series of tablets.Headquartered in Redmond, Washington, Microsoft is the world's largest software company in terms of revenue. It is also considered one of the world's most valuable companies and one of the most successful startup companies of all time by market capitalization, revenue, growth and cultural impact. Microsoft was founded by Bill Gates and Paul Allen on April 4, 1975, to develop and sell BASIC interpreters for the Altair 8800 microcomputer. The original name, ""Micro-Soft,"" was from a combination of ""microcompu

In [6]:
driver.close()